# Phase 1 Pipeline Check

This notebook verifies the end-to-end extraction and core feature building pipeline over the master dataset.
It implements exact instructions scoped under DiscoveryRank without leaking ranking logic too early.

In [1]:
import sys
import os
import pandas as pd

# Add src dynamically
sys.path.append(os.path.abspath('../src'))
import data_prep
import session_builder
import relevance_labels
import freshness_features

In [2]:
data_dir = '../data'
log_files = ['log_random_4_22_to_5_08_1k.csv']  # Sample using single random file to execute quickly
sample_size = 50000

print("1. Data Prep...")
df = data_prep.load_and_merge_data(data_dir, log_files, sample_size=sample_size)
print(f"Shape after merge: {df.shape}")
print(f"Columns present: {df.columns.tolist()}\n")

print("2. Session Builder...")
df = session_builder.assign_sessions(df)
print(f"Unique sessions created over {df['user_id'].nunique()} users: {df['session_id'].nunique()}\n")

print("3. Relevance Labels...")
df = relevance_labels.create_relevance_labels(df)
print("Relevance Target (y_relevant) Distribution:")
print(df['y_relevant'].value_counts(normalize=True).to_dict())
print("\n")

print("4. Freshness Features...")
df = freshness_features.calculate_freshness(df)
print("Item Age summary (days):")
print(df['item_age_days'].describe())

print("\n5. Saving output...")
output_path = '../outputs/pipeline_sample.csv'
df.to_csv(output_path, index=False)
print(f"Saved final pipeline df successfully to {output_path}!\n")

df.head()

1. Data Prep...


Shape after merge: (43028, 30)
Columns present: ['user_id', 'video_id', 'date', 'hourmin', 'time_ms', 'is_click', 'is_like', 'is_follow', 'is_comment', 'is_forward', 'is_hate', 'long_view', 'play_time_ms', 'duration_ms', 'profile_stay_time', 'comment_stay_time', 'is_profile_enter', 'is_rand', 'tab', 'author_id', 'video_type', 'upload_dt', 'upload_type', 'visible_status', 'video_duration', 'server_width', 'server_height', 'music_id', 'music_type', 'tag']

2. Session Builder...


Unique sessions created over 1000 users: 19045

3. Relevance Labels...
Relevance Target (y_relevant) Distribution:
{0: 0.9488937436088128, 1: 0.051106256391187134}


4. Freshness Features...


Item Age summary (days):
count    43028.000000
mean        22.505680
std          4.666312
min         11.194588
25%         19.277269
50%         23.476551
75%         26.336760
max         29.660845
Name: item_age_days, dtype: float64

5. Saving output...


Saved final pipeline df successfully to ../outputs/pipeline_sample.csv!



,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,...,tag,session_id,explicit_positive_any,explicit_negative,implicit_completion_ratio,y_relevant,upload_dt_parsed,upload_time_ms,item_age_ms,item_age_days
0,0,5543,20220430,1800,1651314030792,0,0,0,0,0,...,25,0_1,0,0,0.028704,0,2022-04-11,1.649635e+12,1.678831e+09,19.430912
1,0,1836,20220502,1200,1651466607423,0,0,0,0,0,...,"62,4",0_2,0,0,0.008986,0,2022-04-10,1.649549e+12,1.917807e+09,22.196845
2,0,721,20220502,1700,1651481542743,0,0,0,0,0,...,15,0_3,0,0,0.060477,0,2022-04-10,1.649549e+12,1.932743e+09,22.369708
3,0,442,20220502,1800,1651488577163,0,0,0,0,0,...,"39,68",0_4,0,0,0.124167,0,2022-04-09,1.649462e+12,2.026177e+09,23.451125
4,0,62,20220503,800,1651535940163,0,0,0,0,0,...,20,0_5,0,0,0.008372,0,2022-04-09,1.649462e+12,2.073540e+09,23.999307
